### Este notebook irá montar um tabela com dados dos Municipios Brasileiros e a quantidade de poplulação de acordo com a situação do domicílio: Urbana ou Rural
### A fonte principal serão os dados do IBGE via api SIDRA - Sistema IBGE de Recuperação Automática (https://sidra.ibge.gov.br/)


In [ ]:
import os, sys, requests
from pyspark.sql import functions as F

In [ ]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")


Recupera as informaçõesde população urbana e rural do IBGE via API SIDRA. 

Tabela 9923 - População residente, por situação do domicílio

A URL é definida para buscar os dados de população urbana e rural por município.

In [5]:
url_populacao_urbana_rural = "https://apisidra.ibge.gov.br/values/t/9923/n6/all/v/allxp/p/all/c1/allxt"
try:
    # 1. Faz a requisição HTTP para a API do IBGE
    response = requests.get(url_populacao_urbana_rural)
    response.raise_for_status()  # Garante que a requisição funcionou (Status 200)

    # 2. Converte a resposta bruta para o formato JSON
    populacao_urbana_rural_dados_json = response.json()

    print(len(populacao_urbana_rural_dados_json))
except Exception as e:
    print(e)    

11141


### Tratamento da variável populacao_urbana_rural_dados_json
### As info

In [10]:


cabecalho    = populacao_urbana_rural_dados_json[0]
linhas_dados = populacao_urbana_rural_dados_json[1:]

mapeamento_colunas = {k: v for k, v in cabecalho.items()}
dados_processados = []
for linha in linhas_dados:
    nova_linha = {mapeamento_colunas[chave]: valor for chave, valor in linha.items() if chave in mapeamento_colunas}
    dados_processados.append(nova_linha)

df_populacao_urbana_rural = spark.createDataFrame(dados_processados)

# Separa somente as colunas desejadas.
df_populacao_urbana_rural_ = \
    df_populacao_urbana_rural.select(
         F.col("Ano").alias("ano")
        ,F.col("Município (Código)").alias("codigo_municipio")
        ,F.col("Situação do domicílio (Código)").alias("codigo_situacao_domicilio")
        ,F.col("Situação do domicílio").alias("descricao_situacao_domicilio")
        ,F.col("Valor").alias("quantidade_populacao_situacao"))



### Faz a transposição das informações de linhas para colunas 

In [15]:
df_populacao_urbana_rural_.createOrReplaceTempView("temp_municipios")
query = \
    """Select ano, codigo_municipio, descricao_situacao_domicilio, quantidade_populacao_situacao
         from temp_municipios
    """
df_municipio_populacao_urbana_rural = spark.sql(query)

# df_municipio_populacao_urbana_rural.printSchema()

# df_municipio_populacao_urbana_rural.show(10,False)

# Executando o Pivot para criar as colunas "Urbana" e "Rural" com os valores correspondentes
df_munic_pivot_situacao = \
    df_municipio_populacao_urbana_rural \
        .groupBy("ano", "codigo_municipio" ) \
        .pivot("descricao_situacao_domicilio", ["Urbana", "Rural"]) \
        .agg(F.first("quantidade_populacao_situacao"))

# Exibindo o resultado
df_munic_pivot_situacao.show(truncate=False)


+----+----------------+------+-----+
|ano |codigo_municipio|Urbana|Rural|
+----+----------------+------+-----+
|2022|1100015         |12971 |8523 |
|2022|1100023         |83952 |12881|
|2022|1100031         |2846  |2505 |
|2022|1100049         |71907 |14980|
|2022|1100056         |14116 |1774 |
|2022|1100064         |11709 |3954 |
|2022|1100072         |2780  |4739 |
|2022|1100080         |6670  |5957 |
|2022|1100098         |21452 |7962 |
|2022|1100106         |31737 |7650 |
|2022|1100114         |39807 |10784|
|2022|1100122         |112983|11350|
|2022|1100130         |16634 |14073|
|2022|1100148         |7309  |8370 |
|2022|1100155         |27701 |7343 |
|2022|1100189         |28710 |6369 |
|2022|1100205         |426299|34135|
|2022|1100254         |12381 |6946 |
|2022|1100262         |1438  |2033 |
|2022|1100288         |47264 |9142 |
+----+----------------+------+-----+
only showing top 20 rows


In [16]:
df_munic_pivot_situacao.printSchema()

root
 |-- ano: string (nullable = true)
 |-- codigo_municipio: string (nullable = true)
 |-- Urbana: string (nullable = true)
 |-- Rural: string (nullable = true)



In [33]:
# Este dataframe contém informações sobre o bioma predominante, a grande região, e a população urbana e rural de cada município. 
# A junção é feita com base no código do município, utilizando um join à esquerda para garantir que todos os municípios do DataFrame de biomas sejam mantidos, 
# mesmo que não existam dados de população urbana/rural correspondentes.

df_munic_sit_bioma = \
    (df_bioma_grande_regiao.alias('b')
        .join(df_munic_pivot_situacao.alias('s')
             ,F.col('b.codigo_municipio') == F.col('s.codigo_municipio')
             ,"left")
        .select('b.codigo_municipio'
               ,'b.municipio'
               ,'b.UF_municipio'
               ,'b.bioma'
               ,'b.grande_regiao'
               ,'s.Urbana'
               ,'s.Rural') )
        
df_munic_sit_bioma.show(truncate=False)

#   .where("s.codigo_municipio is null or (s.Urbana = '0' or s.Rural = '0')") \



+----------------+------------------------+------------+--------+-------------+------+-----+
|codigo_municipio|municipio               |UF_municipio|bioma   |grande_regiao|Urbana|Rural|
+----------------+------------------------+------------+--------+-------------+------+-----+
|1100114         |Jaru                    |RO          |Amazônia|Norte        |39807 |10784|
|1100288         |Rolim de Moura          |RO          |Amazônia|Norte        |47264 |9142 |
|1100189         |Pimenta Bueno           |RO          |Amazônia|Norte        |28710 |6369 |
|1100049         |Cacoal                  |RO          |Amazônia|Norte        |71907 |14980|
|1100262         |Rio Crespo              |RO          |Amazônia|Norte        |1438  |2033 |
|1100155         |Ouro Preto do Oeste     |RO          |Amazônia|Norte        |27701 |7343 |
|1100205         |Porto Velho             |RO          |Amazônia|Norte        |426299|34135|
|1100064         |Colorado do Oeste       |RO          |Amazônia|Norte

In [34]:
# Salva o Dataframe como csv
(df_munic_sit_bioma
    .toPandas()
    .to_csv(r"C:\Marco Conti\Projetos\MAIS-v2\dados\tb_munic_sit_bioma.csv"
           ,index=False
           ,sep=";"))



c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


In [ ]:
# import subprocess

# resultado = subprocess.run(
#     r'"c:\Marco Conti\Projetos\MAIS-v2\.venv\Scripts\python.exe" -m pip install PyArrow', # trocar show por install 
#     shell=True,
#     capture_output=True,
#     text=True
# )

# print(resultado.stdout)
# print(resultado.stderr)
# print(resultado.returncode)

  Using cached pyarrow-25.0.0-cp311-cp311-win_amd64.whl.metadata (3.0 kB)
Using cached pyarrow-25.0.0-cp311-cp311-win_amd64.whl (27.8 MB)


0
